In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType
)
from datetime import datetime
import uuid

# ============================================================
# CONFIGURATION
# ============================================================

storage_account = "stsupplychain2026xx"

pipeline_runs_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "quality/pipeline_runs/"
)

pipeline_name = "Supply Chain Intelligence Pipeline"

# ============================================================
# PATHS DES TABLES PRINCIPALES
# ============================================================

bronze_orders_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "bronze/orders/"
)

bronze_shipments_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "bronze/shipments/"
)

bronze_events_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "bronze/delivery_events/"
)

silver_orders_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "silver/orders/"
)

silver_shipments_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "silver/shipments/"
)

silver_events_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "silver/delivery_events/"
)

# ============================================================
# IDENTIFIANT DU RUN
# ============================================================

run_id = str(uuid.uuid4())

start_time = datetime.now()

# ============================================================
# MÉTRIQUES SIMPLES
# ============================================================

bronze_orders_count = (
    spark.read.format("delta")
    .load(bronze_orders_path)
    .count()
)

bronze_shipments_count = (
    spark.read.format("delta")
    .load(bronze_shipments_path)
    .count()
)

bronze_events_count = (
    spark.read.format("delta")
    .load(bronze_events_path)
    .count()
)

silver_orders_count = (
    spark.read.format("delta")
    .load(silver_orders_path)
    .count()
)

silver_shipments_count = (
    spark.read.format("delta")
    .load(silver_shipments_path)
    .count()
)

silver_events_count = (
    spark.read.format("delta")
    .load(silver_events_path)
    .count()
)

rows_processed = (
    bronze_orders_count
    + bronze_shipments_count
    + bronze_events_count
)

rows_valid = (
    silver_orders_count
    + silver_shipments_count
    + silver_events_count
)

rows_rejected = (
    rows_processed
    - rows_valid
)

end_time = datetime.now()

status = "SUCCESS"

# ============================================================
# CRÉATION DE LA LIGNE DE MONITORING
# ============================================================

schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("pipeline_name", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("rows_processed", LongType(), False),
    StructField("rows_rejected", LongType(), False),
    StructField("status", StringType(), False)
])

run_data = [
    (
        run_id,
        pipeline_name,
        start_time,
        end_time,
        rows_processed,
        rows_rejected,
        status
    )
]

run_df = spark.createDataFrame(
    run_data,
    schema=schema
)

# ============================================================
# ÉCRITURE DELTA
# ============================================================

(
    run_df
    .write
    .format("delta")
    .mode("append")
    .save(pipeline_runs_path)
)

print("Pipeline run enregistré.")

display(run_df)